<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sinhala-wsd/blob/extraction-pipeline/Extraction_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Notebook setup - drive, HF login, paths, seed

In [1]:
!pip install -q datasets huggingface_hub openpyxl

from google.colab import drive, userdata
from huggingface_hub import login, HfApi
import os

drive.mount('/content/drive')
login(token=userdata.get('HF_TOKEN'))

BASE        = "/content/drive/MyDrive/Research-WSD-New"
OUT         = f"{BASE}/extraction_v3"
os.makedirs(OUT, exist_ok=True)

DATASET     = "sinhala-nlp/sinhala-1.5B-corpus"
REVISION    = HfApi().dataset_info(DATASET).sha
RANDOM_SEED = 42
RESERVOIR_K = 1000   # sentences kept per word per source during the scan
print("Dataset revision:", REVISION)

Mounted at /content/drive
Dataset revision: 352f711af658137be6acceea60063c9e711bf4a4


2. Load target words

In [2]:
import re, unicodedata
from collections import defaultdict
import openpyxl

WORDS_XLSX  = f"{BASE}/WSD_Polysemy_Words.xlsx"
WORDS_SHEET = "target_words"
EXPECTED_N  = 65

JOINERS = re.compile(r"[\u200C\u200D]")
def match_key(tok):
    """Key used ONLY for matching: NFC + ZWJ/ZWNJ removed, so ප්\u200Dර and ප්ර match the same."""
    return JOINERS.sub("", unicodedata.normalize("NFC", str(tok)))

def load_target_words(path=WORDS_XLSX, sheet=WORDS_SHEET):
    """-> words (sheet order), {word: No}, {word: [spellings]}."""
    rows = openpyxl.load_workbook(path, read_only=True)[sheet].iter_rows(values_only=True)
    header = [str(h).strip() if h else "" for h in next(rows)]
    i_no, i_word = header.index("No"), header.index("Word")
    words, numbers, spellings, seen_no = [], {}, {}, set()
    for r in rows:
        r = tuple(r) + (None,) * len(header)
        if not r[i_word]:
            continue
        w, no = str(r[i_word]).strip(), int(float(r[i_no]))
        if w in numbers or no in seen_no:
            raise ValueError(f"Duplicate word or number in sheet: {no} {w}")
        words.append(w); numbers[w] = no; seen_no.add(no)
        spellings[w] = [s.strip() for s in re.split(r"[/,]", w) if s.strip()]
    return words, numbers, spellings

TARGETS, word_no, word_spellings = load_target_words()
assert len(TARGETS) == EXPECTED_N, f"Expected {EXPECTED_N} words, got {len(TARGETS)}"

MATCH = {}
for w in TARGETS:
    for s in word_spellings[w]:
        k = match_key(s)
        if k in MATCH and MATCH[k][0] != w:
            raise ValueError(f"Spelling {s} belongs to two words: {MATCH[k][0]} and {w}")
        MATCH[k] = (w, s)
print(f"{len(TARGETS)} words | {len(MATCH)} spellings")
print(TARGETS)

65 words | 78 spellings
['දිය', 'දද', 'නා', 'ගත / ගත්', 'පස / පස්', 'මුව', 'වත / වත්', 'අඹ', 'තෙර', 'දන', 'ලිය', 'වණ', 'උර', 'තල', 'නල', 'පෙළ', 'මද', 'මී', 'රැස', 'වර', 'වැට', 'අත', 'ගල', 'උත්තර', 'කඩ', 'කම්', 'කලාපය', 'සුර', 'බල', 'නද', 'සත්', 'සල් / සල', 'පත / පත්', 'සක', 'පැහ, පැහැ', 'තුරා / තුරු', 'තිර', 'සඳ', 'සිළ / සිළි / සිළු', 'නුබ, නුඹ', 'හස / හස්', 'නිදන', 'පණ', 'පිය', 'දත්', 'කලි', 'පට', 'රද', 'දෙස් / දෙස', 'සමර', 'වස්', 'සර', 'කුළු', 'කර', 'කමල', 'කන', 'දීප', 'රජ', 'රස', 'රා', 'වැඩ', 'දල', 'ලේ', 'සී', 'කල/කල්']


### 3. Cleaning, Sentence Splitting, Scanner with Checkpoints
3.1. Cleaning
- NFC; URLs and zero-width characters removed; split at line breaks and `. ! ? ।`.
- A sentence is kept if it has 15–400 characters, ≥ 50 % Sinhala characters and ≥ 4 Sinhala tokens (same minimum as the textbook), and is not a date/time line.
- `dedup_key`: the sentence's Sinhala letters and digits only. It is used for exact deduplication per word. Near-duplicates are not removed, because one corpus is used.

In [3]:
import random, hashlib, pickle, time
from collections import Counter
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset

URL_PATTERN      = re.compile(r"https?://\S+|www\.\S+")
SINHALA_CHAR     = re.compile(r"[\u0D80-\u0DFF]")
NOISE_CHARS      = re.compile(r"[\u00AD\u200B\uFEFF]")
LINE_SPLIT       = re.compile(r"[\r\n]+")
SENTENCE_SPLIT   = re.compile(r"(?<=[.!?\u0964])\s*")
METADATA_PATTERN = re.compile(
    r"(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4}"
    r"|\d{1,2}:\d{2}\s*(AM|PM)", re.IGNORECASE)
TOKEN_PATTERN    = re.compile(r"[\u0D80-\u0DFF\u200C\u200D]+")
MIN_CHARS, MAX_CHARS, MIN_SINHALA_RATIO, MIN_TOKENS = 15, 400, 0.5, 4

def segment(text):
    """Document -> clean sentences."""
    text = URL_PATTERN.sub("", NOISE_CHARS.sub("", unicodedata.normalize("NFC", text or "")))
    out = []
    for line in LINE_SPLIT.split(text):
        for s in SENTENCE_SPLIT.split(re.sub(r"\s+", " ", line).strip()):
            s = s.strip()
            if not (MIN_CHARS <= len(s) <= MAX_CHARS):
                continue
            if len(SINHALA_CHAR.findall(s)) / len(s) < MIN_SINHALA_RATIO or METADATA_PATTERN.search(s):
                continue
            if len(TOKEN_PATTERN.findall(s)) < MIN_TOKENS:
                continue
            out.append(s)
    return out

def dedup_key(s):
    return re.sub(r"[^\u0D80-\u0DFF0-9]", "", match_key(s))

### 3.2. Word-level filters
- Keys are **spellings**, so they don't depend on how the Word cell is written.
- A sentence is dropped for a word only if **every** occurrence of the word sits inside a pattern.
- `NOISE`: the token is not the target word at all (boilerplate, abbreviations, initials, a split compound, misspellings).
- `GRAMMATICAL`: the target is used only as a function word.Use the same dict in the textbook notebook. so both sources are filtered identically.
- The patterns are starting points. Check them on the pilot (4.1) before the full run.

In [4]:
S = r"[\u0D80-\u0DFF\u200C\u200D]"          # one Sinhala character (for word boundaries)
NOISE = {
    "රැස": [r"ඇස\s*පාදන\s*රැස"],                                         # website column title
    "සල්": [r"\(\s*සල්\s*\)"],                                            # Islamic honorific abbreviation
    "සී":  [rf"(?<!{S})සී\s*\.", r"විටමින්\s*සී"],                       # English letter C (initials, vitamin C)
    "මී":  [r"මී\s*ළඟ", r"මී\s*ළග"],                                     # split මීළඟ ('next')
    "දන":  [r"දන\s*(?:ගහ|ගැහ|ගැස|ගසා|ගස්ස)"],                         # misspelt දණ ගහනවා ('kneel')
}
GRAMMATICAL = {
    "කර":  [rf"කර\s*(?:ඇත|ඇති|තිබේ|තිබුණි|තිබෙන|තිබූ|ගැනීම|ගන්න|ගත්|ගෙන|දී|සිටි|සිටින|නැත)(?!{S})"],  # 'කර ඇත' = has done
    "කලි": [r"වූ\s*කලි"],                                                 # 'වූ කලි' = when
    "වස්": [r"නු\s*වස්"],                                                 # 'කරනු වස්' = in order to
}

def _compile(table):
    out = {}
    for spelling, pats in table.items():
        k = match_key(spelling)
        assert k in MATCH, f"Filter key '{spelling}' is not a spelling in the word list"
        out.setdefault(MATCH[k][0], []).extend(re.compile(p) for p in pats)
    return out
FILTERS = {"noise": _compile(NOISE), "grammatical": _compile(GRAMMATICAL)}

def reject_reason(word, sent, n_hits):
    """'noise' / 'grammatical' if every occurrence of the word is inside such a pattern, else None."""
    for reason, table in FILTERS.items():
        pats = table.get(word)
        if pats and sum(len(p.findall(sent)) for p in pats) >= n_hits:
            return reason
    return None

### 3.2 Scanner, checkpointed extraction and outputs
- `Scanner.add`: matches whole tokens against the spellings, skips exact duplicates, applies the filters, then keeps a reservoir sample.
- Reservoir sampling (Algorithm R, Vitter 1985) gives an unbiased random sample of up to `RESERVOIR_K` sentences per word from one pass.
- `extract`: streams the corpus and saves a checkpoint every 100,000 documents; re-running resumes. A document with no `source` is counted as `unlabelled`.
- `save_outputs`: pool CSV with provenance, a summary workbook, and `meta.json`

In [5]:
class Scanner:
    """Reservoir sample (Algorithm R) of up to k sentences per word, plus counts of all distinct
    matching sentences per word / spelling / sub-corpus, and rejection counts + examples."""
    def __init__(self, name, k=RESERVOIR_K, seed=RANDOM_SEED):
        self.name, self.k, self.rng = name, k, random.Random(seed)
        self.pool, self.seen = defaultdict(list), defaultdict(set)
        self.n, self.spell_n, self.sub_n, self.rejected = Counter(), Counter(), Counter(), Counter()
        self.rejected_examples = defaultdict(list)

    def add(self, sent, h, sub_source, doc_id):
        hits, first = Counter(), {}
        for tok in TOKEN_PATTERN.findall(sent):
            m = MATCH.get(match_key(tok))
            if m:
                hits[m[0]] += 1; first.setdefault(m[0], (tok, m[1]))
        for word, n_hits in hits.items():
            if h in self.seen[word]:
                continue
            self.seen[word].add(h)
            reason = reject_reason(word, sent, n_hits)
            if reason:
                self.rejected[(word, reason)] += 1
                if len(self.rejected_examples[(word, reason)]) < 20:
                    self.rejected_examples[(word, reason)].append(sent)
                continue
            tok, spelling = first[word]
            self.n[word] += 1; self.spell_n[(word, spelling)] += 1; self.sub_n[(word, sub_source)] += 1
            row = {"No": word_no[word], "word": word, "matched": tok, "source": self.name,
                   "sub_source": sub_source, "doc_id": doc_id, "sentence": sent}
            pool = self.pool[word]
            if len(pool) < self.k:
                pool.append(row)
            else:
                j = self.rng.randint(1, self.n[word])
                if j <= self.k:
                    pool[j - 1] = row

def extract(run, dataset, text_key="text", sub_key="source", id_key=None, max_docs=None, every=100_000):
    """Scan the corpus ONCE; checkpoint every `every` docs; re-running resumes. One checkpoint per run name."""
    ckpt = f"{OUT}/ckpt_{run}.pkl"
    def save():
        pickle.dump({**state, "scanner": state["scanner"].__dict__}, open(ckpt, "wb"))
    if os.path.exists(ckpt):
        state = pickle.load(open(ckpt, "rb"))
        sc = Scanner.__new__(Scanner); sc.__dict__.update(state["scanner"]); state["scanner"] = sc
        print(f"Resuming {run} after {state['docs_done']:,} documents")
        dataset = dataset.skip(state["docs_done"])
    else:
        state = {"docs_done": 0, "docs": Counter(), "scanner": Scanner(run), "minutes": 0.0}
    t0 = time.time()
    for rec in tqdm(dataset, desc=run, initial=state["docs_done"]):
        if max_docs and state["docs_done"] >= max_docs:
            break
        sub = rec.get(sub_key) or "unlabelled"          # never relabel missing sources as the corpus name
        doc_id = rec.get(id_key) if id_key else state["docs_done"]
        state["docs_done"] += 1; state["docs"][sub] += 1
        for sent in segment(rec.get(text_key)):
            h = int.from_bytes(hashlib.blake2b(dedup_key(sent).encode(), digest_size=8).digest(), "big")
            state["scanner"].add(sent, h, sub, doc_id)
        if state["docs_done"] % every == 0:
            save()
    state["minutes"] += (time.time() - t0) / 60
    save()
    print(f"{run}: {state['docs_done']:,} docs, {state['minutes']:.1f} min | per sub-corpus: {dict(state['docs'])}")
    save_outputs(state["scanner"], state)
    return state["scanner"]

def save_outputs(sc, state):
    """Pool CSV (with provenance), per-word summary workbook and metadata JSON."""
    tag = f"{OUT}/{sc.name}"
    df = pd.DataFrame([r for w in TARGETS for r in sc.pool[w]],
                      columns=["No", "word", "matched", "source", "sub_source", "doc_id", "sentence"])
    df.to_csv(f"{tag}_pool.csv", index=False, encoding="utf-8")
    summary = pd.DataFrame([{
        "No": word_no[w], "word": w, "matches_kept": sc.n[w], "in_pool": len(sc.pool[w]),
        "rejected_noise": sc.rejected[(w, "noise")], "rejected_grammatical": sc.rejected[(w, "grammatical")],
        "by_spelling": ", ".join(f"{s} ({c})" for (ww, s), c in sc.spell_n.most_common() if ww == w),
        "by_sub_corpus": ", ".join(f"{s} ({c})" for (ww, s), c in sc.sub_n.most_common() if ww == w),
    } for w in TARGETS])
    with pd.ExcelWriter(f"{tag}_summary.xlsx") as xw:
        summary.to_excel(xw, sheet_name="per_word", index=False)
        pd.DataFrame([{"sub_corpus": k, "documents": v} for k, v in state["docs"].items()]
                     ).to_excel(xw, sheet_name="documents", index=False)
    meta = {
        "corpus": DATASET, "revision": REVISION, "run_name": sc.name, "date": time.strftime("%Y-%m-%d"),
        "citation": "Ranasinghe et al. (2025). Sinhala Encoder-only Language Models and Evaluation. ACL 2025.",
        "word_list": os.path.basename(WORDS_XLSX), "word_sheet": WORDS_SHEET, "n_words": len(TARGETS),
        "matching": "base form only; any listed spelling = same word; whole tokens; NFC; ZWJ/ZWNJ-insensitive",
        "cleaning": {"min_chars": MIN_CHARS, "max_chars": MAX_CHARS, "min_sinhala_ratio": MIN_SINHALA_RATIO,
                     "min_tokens": MIN_TOKENS, "urls_removed": True, "date_time_lines_removed": True},
        "dedup": "exact, per word (hash of Sinhala letters + digits); no near-duplicate removal",
        "sampling": {"method": "reservoir (Algorithm R)", "k_per_word": sc.k, "seed": RANDOM_SEED},
        "noise_patterns": NOISE, "grammatical_patterns": GRAMMATICAL,
        "documents": state["docs_done"], "documents_per_sub_corpus": dict(state["docs"]),
        "scan_minutes": round(state["minutes"], 1),
        "matches_kept_total": int(summary.matches_kept.sum()),
        "rejected": {f"{w} | {r}": c for (w, r), c in sc.rejected.items()},
        "words_with_0": list(summary.loc[summary.matches_kept == 0, "word"]),
    }
    json.dump(meta, open(f"{tag}_meta.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"  [{sc.name}] {len(df):,} sentences in pool | words with 0: {meta['words_with_0']}")

In [ ]:
ds = load_dataset("sinhala-nlp/sinhala-1.5B-corpus", split="train", streaming=True, token=True)
c15_base = extract("corpus15b", ["base"], ds, text_key="text", sub_key="source")

README.md:   0%|          | 0.00/7.90k [00:00<?, ?B/s]

corpus15b [base]: 0it [00:00, ?it/s]

corpus15b: 7,074,641 docs in 349.4 min | docs per sub-corpus: {'hplt_2': 1152703, 'CulturaX': 753655, 'Nsina': 486932, 'sinmin': 313910, 'corpus15b': 3895829, 'FacebookDecadeCorpora': 364402, 'semisold': 107210}
  [corpus15b_target_word_only] 65,713 sentences saved | 9,262,215 matches in corpus | words with 0: []


In [ ]:
import os, random
import pandas as pd

N_PER_WORD = 200
SEED       = 42
# use the re-run with correct source labels if it exists, otherwise the first run
SRC = next(p for p in [f"{OUT}/corpus15b_v2_target_word_only_sentences.csv",
                       f"{OUT}/corpus15b_target_word_only_sentences.csv"] if os.path.exists(p))
print("Reading", SRC)

pool = pd.read_csv(SRC)
pool = pool.drop_duplicates(subset=["word", "sentence"])          # safety: one copy of a sentence per word

rng, parts, rows = random.Random(SEED), [], []
for w in TARGETS:                                                 # the 67 words, in sheet order
    g = pool[pool.word == w]
    take = g.sample(n=min(N_PER_WORD, len(g)), random_state=rng.randint(0, 2**31 - 1)) if len(g) else g
    parts.append(take)
    rows.append({"No": word_no[w], "word": w, "available": len(g), "sampled": len(take),
                 "short_of_200": max(0, N_PER_WORD - len(take)),
                 "by_spelling": ", ".join(f"{k} ({v})" for k, v in take.matched.value_counts().items()),
                 "by_sub_corpus": ", ".join(f"{k} ({v})" for k, v in take.sub_source.value_counts().items())})
sample = pd.concat(parts, ignore_index=True)
summary = pd.DataFrame(rows)

tag = f"{OUT}/target_word_only_{N_PER_WORD}_per_word"
sample.to_csv(f"{tag}.csv", index=False, encoding="utf-8")
with open(f"{tag}.txt", "w", encoding="utf-8") as f:              # annotation-tool format
    f.write("\n\n".join(f"s_word: {w}\nsentences: "
                        + "; ".join(s.replace(";", ",") for s in sample.loc[sample.word == w, "sentence"])
                        for w in TARGETS))
summary.to_excel(f"{tag}_summary.xlsx", index=False)

print(f"{len(sample):,} sentences for {sample.word.nunique()} words (planned {N_PER_WORD * len(TARGETS):,})")
short = summary[summary.short_of_200 > 0]
print("Words with fewer than 200:" if len(short) else "Every word has 200.",
      short[["word", "available"]].to_string(index=False) if len(short) else "")

Reading /content/drive/MyDrive/Research-WSD-New/extraction_v3/corpus15b_target_word_only_sentences.csv


NameError: name 'Words' is not defined